In [1]:
from pathlib import Path
import pandas as pd
import ollama

ROOT = Path.home() / "Projects" / "support-copilot"
df = pd.read_csv(ROOT / "data" / "support_data.csv")

CATEGORIES = sorted(df["category"].unique())
print(len(CATEGORIES), "categories:", CATEGORIES)

11 categories: ['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


In [2]:
def route(message, model="llama3.2:3b"):
    prompt = f"""You are a customer support ticket router.
Classify the customer message into exactly ONE of these categories:
{", ".join(CATEGORIES)}

Reply with ONLY the category name. No explanation.

Customer message: {message}
Category:"""

    reply = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    answer = reply["message"]["content"].strip().upper()

    # Find which allowed category the model mentioned
    matches = [c for c in CATEGORIES if c in answer]
    if matches:
        return max(matches, key=len)
    return "UNKNOWN"

In [3]:
route("hey i was charged twice for my last order, i want my money back!!")

'REFUND'

In [4]:
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    predicted = route(row["instruction"])
    correct = row["category"]
    mark = "✅" if predicted == correct else "❌"
    print(f"{mark} {row['instruction'][:70]}")
    print(f"   predicted: {predicted} | correct: {correct}\n")

✅ I can't talk with  a human agent
   predicted: CONTACT | correct: CONTACT

❌ I have got to locate hte bills from {{Person Name}}
   predicted: ACCOUNT | correct: INVOICE

✅ I cannot pay, help me to inform of a problem with payments
   predicted: PAYMENT | correct: PAYMENT

✅ I want help speaking to customer service
   predicted: CONTACT | correct: CONTACT

❌ I try to see th accepted payment options
   predicted: ORDER | correct: PAYMENT



In [5]:
# 9 random messages from each of the 11 categories = 99 test tickets
test = df.groupby("category").sample(n=9, random_state=42).reset_index(drop=True)

print(len(test), "test tickets")
print(test["category"].value_counts())

test.to_csv(ROOT / "data" / "test_set.csv", index=False)

99 test tickets
category
ACCOUNT         9
CANCEL          9
CONTACT         9
DELIVERY        9
FEEDBACK        9
INVOICE         9
ORDER           9
PAYMENT         9
REFUND          9
SHIPPING        9
SUBSCRIPTION    9
Name: count, dtype: int64


In [6]:
import time

predictions = []
times = []

for i, message in enumerate(test["instruction"]):
    start = time.perf_counter()
    predictions.append(route(message))
    times.append(time.perf_counter() - start)
    if (i + 1) % 10 == 0:
        print(f"{i + 1}/{len(test)} done")

test["predicted"] = predictions
test["seconds"] = times
print("Finished ✅")

10/99 done
20/99 done
30/99 done
40/99 done
50/99 done
60/99 done
70/99 done
80/99 done
90/99 done
Finished ✅


In [7]:
test["correct"] = test["predicted"] == test["category"]

accuracy = test["correct"].mean() * 100
unknown = (test["predicted"] == "UNKNOWN").sum()
avg_time = test["seconds"].mean()

print(f"Accuracy: {accuracy:.1f}%  ({test['correct'].sum()}/{len(test)})")
print(f"Off-menu answers (UNKNOWN): {unknown}")
print(f"Average time per ticket: {avg_time:.2f} s")

Accuracy: 38.4%  (38/99)
Off-menu answers (UNKNOWN): 0
Average time per ticket: 0.17 s


In [9]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 50.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 43.1 MB/s  0:00:005.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [10]:
from sklearn.metrics import classification_report

print(classification_report(test["category"], test["predicted"], zero_division=0))

              precision    recall  f1-score   support

     ACCOUNT       0.17      0.33      0.22         9
      CANCEL       0.00      0.00      0.00         9
     CONTACT       0.27      1.00      0.43         9
    DELIVERY       1.00      0.78      0.88         9
    FEEDBACK       0.41      0.78      0.54         9
     INVOICE       0.00      0.00      0.00         9
       ORDER       0.35      0.67      0.46         9
     PAYMENT       1.00      0.44      0.62         9
      REFUND       1.00      0.22      0.36         9
    SHIPPING       0.00      0.00      0.00         9
SUBSCRIPTION       0.00      0.00      0.00         9

    accuracy                           0.38        99
   macro avg       0.38      0.38      0.32        99
weighted avg       0.38      0.38      0.32        99



In [11]:
pd.crosstab(test["category"], test["predicted"], rownames=["correct"], colnames=["predicted"])

predicted,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,ORDER,PAYMENT,REFUND
correct,,,,,,,,
ACCOUNT,3,0,5,0,0,1,0,0
CANCEL,8,0,0,0,0,1,0,0
CONTACT,0,0,9,0,0,0,0,0
DELIVERY,0,0,2,7,0,0,0,0
FEEDBACK,1,0,1,0,7,0,0,0
INVOICE,0,0,1,0,5,3,0,0
ORDER,0,1,2,0,0,6,0,0
PAYMENT,1,0,4,0,0,0,4,0
REFUND,2,0,0,0,5,0,0,2


In [12]:
mistakes = test[~test["correct"]]
print(len(mistakes), "mistakes\n")

for _, row in mistakes.iterrows():
    print(f"❌ {row['instruction']}")
    print(f"   predicted: {row['predicted']} | correct: {row['category']}\n")

61 mistakes

❌ where to notify of problems with sign-up?
   predicted: CONTACT | correct: ACCOUNT

❌ wanna  use the {{Account Type}} profile help me
   predicted: CONTACT | correct: ACCOUNT

❌ I cannot retrieve the bloody PIN code of my profile
   predicted: CONTACT | correct: ACCOUNT

❌ i need assistance changing to the {{Account Type}} account
   predicted: ORDER | correct: ACCOUNT

❌ I have to report signup issues
   predicted: CONTACT | correct: ACCOUNT

❌ i dont know what to do to notify of an issue with a sign-up
   predicted: CONTACT | correct: ACCOUNT

❌ i need help to see the early termination penalty
   predicted: ACCOUNT | correct: CANCEL

❌ I do not know how to check the cancellation  penalties
   predicted: ACCOUNT | correct: CANCEL

❌ how can i check the withdrawal penalty
   predicted: ACCOUNT | correct: CANCEL

❌ I need assistance seeing the early termination penalties
   predicted: ACCOUNT | correct: CANCEL

❌ how to see the cancellation fees?
   predicted: ACCOUNT | c

In [13]:
(ROOT / "results").mkdir(exist_ok=True)
test.to_csv(ROOT / "results" / "v1_zero_shot.csv", index=False)
print("Saved results ✅")

Saved results ✅


In [14]:
print(test["predicted"].value_counts())
print()
print(df.groupby("category")["intent"].unique())

predicted
CONTACT     33
ACCOUNT     18
ORDER       17
FEEDBACK    17
DELIVERY     7
PAYMENT      4
REFUND       2
CANCEL       1
Name: count, dtype: int64

category
ACCOUNT         [create_account, delete_account, edit_account,...
CANCEL                                   [check_cancellation_fee]
CONTACT           [contact_customer_service, contact_human_agent]
DELIVERY                      [delivery_options, delivery_period]
FEEDBACK                                      [complaint, review]
INVOICE                              [check_invoice, get_invoice]
ORDER           [cancel_order, change_order, place_order, trac...
PAYMENT                    [check_payment_methods, payment_issue]
REFUND            [check_refund_policy, get_refund, track_refund]
SHIPPING        [change_shipping_address, set_up_shipping_addr...
SUBSCRIPTION                            [newsletter_subscription]
Name: intent, dtype: object


In [15]:
import time

def evaluate(router, name):
    preds, times = [], []
    for i, msg in enumerate(test["instruction"]):
        start = time.perf_counter()
        preds.append(router(msg))
        times.append(time.perf_counter() - start)
        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(test)}")

    res = test[["instruction", "category"]].copy()
    res["predicted"] = preds
    res["seconds"] = times
    res["correct"] = res["predicted"] == res["category"]

    acc = res["correct"].mean() * 100
    print(f"{name}: accuracy {acc:.1f}% | avg {res['seconds'].mean():.2f}s/ticket | "
          f"UNKNOWN {(res['predicted'] == 'UNKNOWN').sum()}")
    res.to_csv(ROOT / "results" / f"{name}.csv", index=False)
    return res

In [16]:
DEFINITIONS = {
    "ACCOUNT": "creating, deleting, editing or switching an account; recovering a password; registration problems",
    "CANCEL": "asking about the FEE or penalty for cancelling (NOT actually cancelling an order)",
    "CONTACT": "wants to talk to customer service or a human agent, or asks how to contact the company",
    "DELIVERY": "what delivery options exist and how long delivery takes",
    "FEEDBACK": "making a complaint or leaving a review about the company or service",
    "INVOICE": "finding, checking or downloading an invoice or bill",
    "ORDER": "placing, changing, cancelling or tracking an order or purchase",
    "PAYMENT": "which payment methods are accepted, or problems making a payment",
    "REFUND": "refund policy, asking for money back, or tracking a refund",
    "SHIPPING": "setting up or changing the shipping address",
    "SUBSCRIPTION": "subscribing to or unsubscribing from the newsletter",
}

In [17]:
def route_v2(message, model="llama3.2:3b", examples=None):
    definitions = "\n".join(f"- {c}: {d}" for c, d in DEFINITIONS.items())

    shots = ""
    if examples is not None:
        shots = "\nExamples of correctly classified messages:\n"
        for r in examples.itertuples():
            shots += f"Message: {r.instruction}\nCategory: {r.category}\n\n"

    prompt = f"""You are a customer support ticket router.
Classify the customer message into exactly ONE category.

Category definitions:
{definitions}
{shots}
Reply with ONLY the category name. No explanation.

Customer message: {message}
Category:"""

    reply = ollama.chat(model=model,
                        messages=[{"role": "user", "content": prompt}],
                        options={"temperature": 0})
    answer = reply["message"]["content"].strip().upper()
    matches = [c for c in CATEGORIES if c in answer]
    return max(matches, key=len) if matches else "UNKNOWN"

In [18]:
v2 = evaluate(lambda m: route_v2(m), "v2_definitions")

  20/99
  40/99
  60/99
  80/99
v2_definitions: accuracy 78.8% | avg 0.18s/ticket | UNKNOWN 0


In [19]:
# Use only messages that are NOT in the test set
pool = df[~df["instruction"].isin(test["instruction"])]

examples = (pool.groupby("category")
                .sample(n=2, random_state=7)
                .sample(frac=1, random_state=7))   # shuffle the order

print(len(examples), "examples")
examples[["instruction", "category"]]

22 examples


,instruction,category
10346,I don't know how I can create a {{Account Type...,ACCOUNT
8041,how can i talk wiht customer service,CONTACT
3569,where do i see the cancellation penalties,CANCEL
17738,I want assistance to sign up to the newsletter,SUBSCRIPTION
2097,want support trying to upate my shipping address,SHIPPING
25846,ETA of purchase {{Order Number}},ORDER
15593,I do not know how I can get my invoice from {{...,INVOICE
24120,I do not know what to do to change to the {{Ac...,ACCOUNT
17647,can I subscribe to the corporate newsletter?,SUBSCRIPTION
12129,can i order from {{Delivery City}},DELIVERY


In [20]:
v3 = evaluate(lambda m: route_v2(m, examples=examples), "v3_definitions_fewshot")

  20/99
  40/99
  60/99
  80/99
v3_definitions_fewshot: accuracy 98.0% | avg 0.16s/ticket | UNKNOWN 0


In [21]:
summary = pd.DataFrame({
    "version": ["v1 zero-shot", "v2 + definitions", "v3 + definitions + few-shot"],
    "accuracy_%": [test["correct"].mean() * 100, v2["correct"].mean() * 100, v3["correct"].mean() * 100],
    "sec_per_ticket": [test["seconds"].mean(), v2["seconds"].mean(), v3["seconds"].mean()],
}).round(2)
summary

,version,accuracy_%,sec_per_ticket
0,v1 zero-shot,38.38,0.17
1,v2 + definitions,78.79,0.18
2,v3 + definitions + few-shot,97.98,0.16


In [22]:
pd.crosstab(v3["category"], v3["predicted"], rownames=["correct"], colnames=["predicted"])

predicted,ACCOUNT,CANCEL,CONTACT,DELIVERY,FEEDBACK,INVOICE,ORDER,PAYMENT,REFUND,SHIPPING,SUBSCRIPTION
correct,,,,,,,,,,,
ACCOUNT,7,0,1,0,1,0,0,0,0,0,0
CANCEL,0,9,0,0,0,0,0,0,0,0,0
CONTACT,0,0,9,0,0,0,0,0,0,0,0
DELIVERY,0,0,0,9,0,0,0,0,0,0,0
FEEDBACK,0,0,0,0,9,0,0,0,0,0,0
INVOICE,0,0,0,0,0,9,0,0,0,0,0
ORDER,0,0,0,0,0,0,9,0,0,0,0
PAYMENT,0,0,0,0,0,0,0,9,0,0,0
REFUND,0,0,0,0,0,0,0,0,9,0,0


In [23]:
used = set(test["instruction"]) | set(examples["instruction"])
test2 = (df[~df["instruction"].isin(used)]
           .groupby("category").sample(n=9, random_state=123)
           .reset_index(drop=True))

test_backup = test
test = test2                     # evaluate() uses the variable `test`
v3_check = evaluate(lambda m: route_v2(m, examples=examples), "v3_check_new_testset")
test = test_backup               # restore the original

  20/99
  40/99
  60/99
  80/99
v3_check_new_testset: accuracy 93.9% | avg 0.18s/ticket | UNKNOWN 0


In [24]:
v3[~v3["correct"]][["instruction", "category", "predicted"]]

,instruction,category,predicted
0,where to notify of problems with sign-up?,ACCOUNT,CONTACT
7,i dont know what to do to notify of an issue w...,ACCOUNT,FEEDBACK


In [25]:
exclude = set(test["instruction"]) | set(test2["instruction"])
train = df[~df["instruction"].isin(exclude)]
print(len(train), "training messages")

26610 training messages


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000),
)

start = time.perf_counter()
clf.fit(train["instruction"], train["category"])
print(f"Trained in {time.perf_counter() - start:.1f} seconds")

Trained in 0.5 seconds


In [27]:
for name, t in [("test set 1", test), ("test set 2 (fresh)", test2)]:
    start = time.perf_counter()
    preds = clf.predict(t["instruction"])
    per_ticket = (time.perf_counter() - start) / len(t)
    acc = (preds == t["category"]).mean() * 100
    print(f"Classifier on {name}: accuracy {acc:.1f}% | {per_ticket * 1000:.2f} ms/ticket")

Classifier on test set 1: accuracy 98.0% | 0.08 ms/ticket
Classifier on test set 2 (fresh): accuracy 99.0% | 0.03 ms/ticket


In [38]:
msg = "hey my bill from last month is not showing anywhere"
probs = clf.predict_proba([msg])[0]
top = sorted(zip(clf.classes_, probs), key=lambda x: x[1], reverse=True)[:3]
for cat, p in top:
    print(f"{cat}: {p:.0%}")

INVOICE: 80%
ACCOUNT: 5%
ORDER: 2%


In [29]:
clf_acc1 = (clf.predict(test["instruction"]) == test["category"]).mean() * 100
clf_acc2 = (clf.predict(test2["instruction"]) == test2["category"]).mean() * 100

final = pd.DataFrame({
    "approach": ["LLM zero-shot", "LLM + definitions", "LLM + definitions + few-shot",
                 "scikit-learn classifier"],
    "labelled_examples_used": [0, 0, 22, len(train)],
    "acc_test1_%": [38.4, 78.8, 98.0, round(clf_acc1, 1)],
    "acc_test2_%": [None, None, 93.9, round(clf_acc2, 1)],
    "speed": ["~170 ms", "~180 ms", "~160 ms", "<1 ms"],
})
final

,approach,labelled_examples_used,acc_test1_%,acc_test2_%,speed
0,LLM zero-shot,0,38.4,NaN,~170 ms
1,LLM + definitions,0,78.8,NaN,~180 ms
2,LLM + definitions + few-shot,22,98.0,93.9,~160 ms
3,scikit-learn classifier,26610,98.0,99.0,<1 ms


In [39]:
clf_char = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2),
    LogisticRegression(max_iter=1000),
)
clf_char.fit(train["instruction"], train["category"])

for name, t in [("test 1", test), ("test 2", test2)]:
    acc = (clf_char.predict(t["instruction"]) == t["category"]).mean() * 100
    print(f"Char classifier on {name}: {acc:.1f}%")

Char classifier on test 1: 100.0%
Char classifier on test 2: 100.0%


In [40]:
my_msgs = ["need cash back", "need cash bac", "i have to give a item back",
           "i have to give a itm back", "i have to ret a order"]

for m in my_msgs:
    print(f"{m!r}")
    print(f"   word classifier: {clf.predict([m])[0]}")
    print(f"   char classifier: {clf_char.predict([m])[0]}")
    print(f"   LLM (v3):        {route_v2(m, examples=examples)}\n")

'need cash back'
   word classifier: REFUND
   char classifier: REFUND
   LLM (v3):        REFUND

'need cash bac'
   word classifier: ACCOUNT
   char classifier: REFUND
   LLM (v3):        REFUND

'i have to give a item back'
   word classifier: ORDER
   char classifier: ORDER
   LLM (v3):        REFUND

'i have to give a itm back'
   word classifier: REFUND
   char classifier: REFUND
   LLM (v3):        REFUND

'i have to ret a order'
   word classifier: ORDER
   char classifier: ORDER
   LLM (v3):        ORDER



In [41]:
kb = train.groupby("intent").sample(n=3, random_state=1).reset_index(drop=True)
print(len(kb), "knowledge base entries")
kb[["instruction", "intent", "response"]].head(3)

81 knowledge base entries


,instruction,intent,response
0,have a problem with cancelling order {{Order N...,cancel_order,I'm conscious of the reality that you're encou...
1,"I bought some bloody articles, cancel order {{...",cancel_order,I certainly recognize that you're looking to c...
2,I have a question about canceling purchase {{O...,cancel_order,Happy to get your message to us with your ques...


In [42]:
import chromadb

def embed(text):
    return ollama.embed(model="nomic-embed-text", input=text)["embeddings"][0]

client = chromadb.PersistentClient(path=str(ROOT / "chroma_db"))
try:
    client.delete_collection("support_kb")
except Exception:
    pass
col = client.create_collection("support_kb", metadata={"hnsw:space": "cosine"})

col.add(
    ids=[str(i) for i in kb.index],
    documents=kb["response"].tolist(),
    embeddings=[embed(q) for q in kb["instruction"]],
    metadatas=[{"category": r.category, "intent": r.intent, "question": r.instruction}
               for r in kb.itertuples()],
)
print(col.count(), "entries stored ✅")

81 entries stored ✅


In [43]:
def retrieve(message, category=None, k=3):
    where = {"category": category} if category else None
    res = col.query(query_embeddings=[embed(message)], n_results=k, where=where)
    return [
        {"question": m["question"], "intent": m["intent"], "answer": d, "distance": round(dist, 3)}
        for d, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

for hit in retrieve("i was charged twice and want my money back", category="REFUND"):
    print(hit["distance"], "|", hit["intent"], "|", hit["question"])

0.35 | get_refund | i do not know how to get a refund of my money
0.405 | check_refund_policy | want assistance seeing in what cases can I request a refund
0.411 | check_refund_policy | i need help checking in what cases can i ask for refunds


In [44]:
def copilot(message):
    category = route_v2(message, examples=examples)
    hits = retrieve(message, category=category)
    context = "\n\n".join(
        f"Past customer question: {h['question']}\nApproved answer: {h['answer']}" for h in hits
    )
    prompt = f"""You are a friendly, professional customer support agent.
Write a reply to the customer using ONLY the approved answers below as your source of information.

Rules:
- Keep it short: maximum 120 words.
- Do not invent policies, prices, phone numbers, emails or links.
- Keep placeholders like {{{{Order Number}}}} exactly as they are.
- If the approved answers do not cover the question, say you will pass it to a human agent.

Approved answers:
{context}

Customer message: {message}
Reply:"""
    reply = ollama.chat(model="llama3.2:3b",
                        messages=[{"role": "user", "content": prompt}],
                        options={"temperature": 0.2})
    return {"category": category, "sources": hits, "reply": reply["message"]["content"].strip()}

In [47]:
for msg in ["hey i was charged twice for my order and want my money back",
            "how do i download the bill for my last purchase",
            "i want to talk to a real person please"]:
    out = copilot(msg)
    print("CUSTOMER:", msg)
    print("CATEGORY:", out["category"])
    print("SOURCES:", [h["intent"] for h in out["sources"]])
    print("REPLY:", out["reply"], "\n" + "-" * 60)

CUSTOMER: hey i was charged twice for my order and want my money back
CATEGORY: REFUND
SOURCES: ['get_refund', 'get_refund', 'check_refund_policy']
REPLY: I've understood that you're seeking assistance on how to receive a refund for the products you purchased. I apologize for any inconvenience caused, and I'm here to support you through this process. To initiate your refund, I recommend reaching out to our customer support team through our designated channels. They will guide you through the necessary steps and provide you with all the information you need to successfully complete the refund. It's essential to have your purchase details and payment information readily available when contacting them. Rest assured, they will do their best to resolve the issue promptly and ensure that you receive the refund you're entitled to. If you have any other questions or concerns, please don't hesitate to let me know. 
------------------------------------------------------------
CUSTOMER: how do i 

In [48]:
in_scope = ["i was charged twice and want a refund", "where is my invoice",
            "how do i reset my password", "when will my package arrive"]
off_topic = ["what is the weather in barcelona", "who won the football match yesterday",
             "can you write me a poem", "what is the capital of france"]

for label, msgs in [("IN SCOPE", in_scope), ("OFF TOPIC", off_topic)]:
    print(label)
    for m in msgs:
        best = retrieve(m)[0]          # search all categories, take the closest match
        print(f"  {best['distance']:.3f}  {m}")

IN SCOPE
  0.322  i was charged twice and want a refund
  0.296  where is my invoice
  0.264  how do i reset my password
  0.222  when will my package arrive
OFF TOPIC
  0.599  what is the weather in barcelona
  0.606  who won the football match yesterday
  0.450  can you write me a poem
  0.606  what is the capital of france


In [49]:
def needs_human(message):
    prompt = f"""Does this customer message need a HUMAN agent instead of an automated reply?
Answer YES if ANY of these is true:
- the customer is very angry, insulting, or threatening to leave or complain publicly
- it mentions legal action, a lawyer, fraud, a safety issue, or an injury
- it describes several different problems at once

Otherwise answer NO.
Reply with ONLY one word: YES or NO.

Customer message: {message}
Answer:"""
    reply = ollama.chat(model="llama3.2:3b", messages=[{"role": "user", "content": prompt}],
                        options={"temperature": 0})
    return "YES" in reply["message"]["content"].upper()

print(needs_human("where is my invoice"))                                           # expect False
print(needs_human("this is the THIRD time you lost my order, i'm calling my lawyer"))  # expect True

False
True


In [50]:
import re

def check_reply(reply, sources):
    problems = []
    allowed = set()
    for h in sources:
        allowed |= set(re.findall(r"\{\{.*?\}\}", h["answer"]))
    invented = set(re.findall(r"\{\{.*?\}\}", reply)) - allowed
    if invented:
        problems.append(f"invented placeholders: {invented}")
    if len(reply.split()) > 120:
        problems.append(f"too long: {len(reply.split())} words")
    return problems

In [51]:
THRESHOLD = 0.39   # ← replace with the value you choose from cell 35

def copilot_v2(message):
    best = retrieve(message)[0]
    if best["distance"] > THRESHOLD:
        return {"action": "ESCALATE", "reason": "no matching knowledge (off-topic or not covered)",
                "category": None, "reply": None, "sources": []}

    if needs_human(message):
        return {"action": "ESCALATE", "reason": "sensitive or complex: needs a human",
                "category": route_v2(message, examples=examples), "reply": None, "sources": []}

    out = copilot(message)
    problems = check_reply(out["reply"], out["sources"])
    if problems:
        return {**out, "action": "ESCALATE", "reason": "; ".join(problems)}
    return {**out, "action": "REPLY", "reason": "passed all checks"}

for m in ["how do i download my invoice",
          "what the weather in barcelone",
          "third time you lost my order, i'm contacting my lawyer",
          "hey i was charged twice for my order and want my money back"]:
    r = copilot_v2(m)
    print(f"{r['action']:9} | {r['reason']} | {m}")

REPLY     | passed all checks | how do i download my invoice
ESCALATE  | no matching knowledge (off-topic or not covered) | what the weather in barcelone
ESCALATE  | no matching knowledge (off-topic or not covered) | third time you lost my order, i'm contacting my lawyer
ESCALATE  | too long: 123 words | hey i was charged twice for my order and want my money back


In [52]:
examples.to_csv(ROOT / "data" / "fewshot_examples.csv", index=False)
print("Saved", len(examples), "examples")

Saved 22 examples
